# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [8]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [9]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0381


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_c7852de926cdeba9,2026-11-10 06:03:04.911000000,u_2e1d30c7,"{'device': 'ios', 'page': 'product', 'region':...","[{'product_id': 'p_c99d34', 'ts': '2026-11-10 ...","[p_8fbf52, p_5969fa, p_0c3c0c, p_40ec57, p_8d8...","[0.013001, 0.303182, 0.215678, 0.293852, 0.040...",NaN,NaN
1,r_be003a7375e44b6d,2026-11-13 02:04:20.325000000,u_c162ac54,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_73a847', 'ts': '2026-11-13 ...","[p_c88f21, p_3c342d, p_59964d, p_1a7fd6, p_1bb...","[0.015102, 0.082563, 0.011842, 0.026765, 0.001...",NaN,NaN
2,r_e4e8a85dfd159ad5,2026-11-11 01:56:10.590000000,u_7bef0d88,"{'device': 'web', 'page': 'product', 'region':...","[{'product_id': 'p_1d1d3f', 'ts': '2026-11-11 ...","[p_8ca8ed, p_4077a5, p_7a1c91, p_92d45d, p_b82...","[0.218886, 0.009675, 0.008384, 0.038402, 0.020...",NaN,NaN


In [11]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

In [12]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

5000 products with co-views


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [13]:
def baseline(row):
    """The live baseline, imported from starter/recommender.py."""
    return recommend(row.recent_views, best)

def coview(row):
    """Products co-viewed with this session's views (most recent first), then the baseline."""
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

POLICIES = {"baseline": baseline, "coview": coview}

## Evaluation

In [ ]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.

___

Métricas:
- hit_rate@10 = (# clicked products in top 10) / (# observed clicks)
- IPC_ctr_slot1
- SNIPS_ctr_slot1
- slot1_matches: # observations where the new policy's first recommendation matches the logging policy's first recommendation

### Two tower

Idea: entrenar una NN con dos partes fundamentales:
1. Embeddings the los productos
2. Embeddings de los usuarios

Salida: producto vectorial de ambos embeddings = score de este producto para este usuario.

Los embeddings de los productos se precomputan en cuanto se puede.

Los embeddings de los usuarios tienen en cuenta los productos.
- Básicamente los fusionamos en un solo vector (por ejemplo con la media) y alimentamos al MLP junto con los datos del usuario.

- Cold start: podemos intentar usar los productos top populares para usuarios sin historial. Podemos, incluso, dar algunas compras de margen antes de empezar usar su historial.

El problema es el timpo. Como mínimo, habrá que seleccionar las últimas N compras. M
as sofisticado: podemos usar algo como un exponential time decay.

Importante: hay que recordar que no se puede sugerir un producto ya comprado (¿o recientemente comprado?).

Posible approach:

#### Redes recurrentes

Son un posible approach. En resumen: tenemos una red recurrente con pesos compartidos.

- En texto es fácil de entender: "the cat sat on...."
- En productos, es similar: "zapatos", "chaqueta", ...

Para distinguir a un usuario de otro, podemos guardar unos estados para cada usuario. Otra opción sería concatenar el embedding del usuario en cada paso.

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
HIDDEN_1_DIM = 256
EMBEDDING_DIMENSIONS = 128

#### ProductTower

In [ ]:
# Red neuronal para mapear un item a un embedding
class ProductTower(nn.Module):
    def __init__(
        self,

        num_categories=10,      # Supuestamente hay 10 (según el README)
        num_subcategories=90,   # Supuestamente hay 90 (según el README)
        num_brands=128,         # Supuestamente hay 80

        category_embedding_dim=16,
        subcategory_embedding_dim=32,
        brand_embedding_dim=32,

        hidden_dim=HIDDEN_1_DIM,
        output_dim=EMBEDDING_DIMENSIONS,
    ):
        super().__init__()

        # Categorical embeddings
        self.category_embedding = nn.Embedding(num_categories, category_embedding_dim)

        self.subcategory_embedding = nn.Embedding(num_subcategories, subcategory_embedding_dim)

        self.brand_embedding = nn.Embedding(num_brands, brand_embedding_dim)

        # Input dimensions: sum
        # 16 + 32 + 32 + 1 (price)
        input_dim = (
            category_embedding_dim
            + subcategory_embedding_dim
            + brand_embedding_dim
            + 1
        )

        # Neural network
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, category, subcategory, brand, price):

        # Categorical embeddings
        category_emb = self.category_embedding(category)
        subcategory_emb = self.subcategory_embedding(subcategory)
        brand_emb = self.brand_embedding(brand)

        # Numerical feature
        price = price.unsqueeze(-1)

        # Price normalization
        price = torch.log1p(price)
        # TODO: price = (price - price_mean) / (price_std + 1e-8)
            # mean y std se deben precomputar usando todos los datos de entrenamiento,
                # y pasar como parámetros al objeto instanciado de la clase

        # Concatenate all features
        x = torch.cat(
            [
                category_emb,
                subcategory_emb,
                brand_emb,
                price
            ],
            dim=1
        )

        # Generate output embedding
        embedding = self.network(x)

        embedding = F.normalize(embedding, p=2, dim=1)

        return embedding


In [ ]:
# TODO: una vez entrenado todo el sistema
    # Intentar comparar objetos que sean claramente similares y objetos claramente distintos

# similarity = F.cosine_similarity(
#     embedding_product_a,
#     embedding_product_b,
#     dim=1
# )

#### UserTower

Alternativa (por ahora descartada):

In [ ]:
# class UserTower(nn.Module):
#     def __init__(
#         self,

#         num_devices,
#         num_regions,
#         num_pages,

#         device_embedding_dim=8,     # Supuestamente hay 3
#         region_embedding_dim=16,    # Supuestamente hay 10
#         page_embedding_dim=8,       # Supuestamente hay 3

#         product_embedding_dim=EMBEDDING_DIMENSIONS,

#         hidden_dim=HIDDEN_1_DIM,
#         output_dim=EMBEDDING_DIMENSIONS,

#         half_life_days=7.0,
#         max_history_days=30.0,
#     ):
#         super().__init__()

#         # Context embeddings
#         self.device_embedding = nn.Embedding(
#             num_devices, device_embedding_dim
#         )

#         self.region_embedding = nn.Embedding(
#             num_regions, region_embedding_dim
#         )

#         self.page_embedding = nn.Embedding(
#             num_pages, page_embedding_dim
#         )

#         self.product_embedding_dim = product_embedding_dim

#         # Temporal decay configuration
#         self.half_life_days = half_life_days
#         self.max_history_days = max_history_days

#         # Event weights:
#         # 0 = view
#         # 1 = add_to_cart
#         # 2 = purchase
#         self.register_buffer(
#             "event_weights",
#             torch.tensor([1.0, 3.0, 5.0])
#         )

#         # Fusion network
#         input_dim = (
#             device_embedding_dim
#             + region_embedding_dim
#             + page_embedding_dim
#             + product_embedding_dim
#         )

#         self.network = nn.Sequential(
#             nn.Linear(input_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Linear(hidden_dim, output_dim)
#         )

#     def forward(
#         self,
#         device,
#         region,
#         page,
#         history_embeddings,
#         history_event_types,
#         history_age_days,
#         history_mask=None,
#     ):

#         # --------------------------------
#         # 1. Context embeddings
#         # --------------------------------

#         device_emb = self.device_embedding(device)
#         region_emb = self.region_embedding(region)
#         page_emb = self.page_embedding(page)

#         # --------------------------------
#         # 2. Temporal weighted pooling
#         # --------------------------------

#         # history_embeddings: (B, N, 128)
#         # history_event_types: (B, N)
#         # history_age_days: (B, N)

#         # Base weight according to event type
#         event_weights = self.event_weights[
#             history_event_types
#         ]

#         # Temporal decay
#         time_weights = torch.pow(
#             2.0,
#             -history_age_days / self.half_life_days
#         )

#         # Combined weights
#         weights = event_weights * time_weights

#         # Ignore old interactions
#         valid = history_age_days <= self.max_history_days

#         # Optional padding mask
#         if history_mask is not None:
#             valid = valid & history_mask.bool()

#         weights = weights * valid.float()

#         # Weighted average of product embeddings
#         weighted_embeddings = (
#             history_embeddings * weights.unsqueeze(-1)
#         )

#         interest_embedding = (
#             weighted_embeddings.sum(dim=1)
#             / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
#         )

#         # --------------------------------
#         # 3. Context + interests
#         # --------------------------------

#         x = torch.cat(
#             [
#                 device_emb,
#                 region_emb,
#                 page_emb,
#                 interest_embedding
#             ],
#             dim=1
#         )

#         # --------------------------------
#         # 4. Generate user embedding
#         # --------------------------------

#         embedding = self.network(x)

#         embedding = F.normalize(
#             embedding,
#             p=2,
#             dim=1
#         )

#         return embedding
